# Laboratorio 3 — Secuencias: RNN y LSTM

**Universidad del Valle**
**Deep Learning y Sistemas Inteligentes — 2026**
**Kevin Recinos**

---

### Caso

El área de monitoreo transaccional de **Banco del Altiplano** quiere pasar de un sistema de reglas
sobre variables agregadas (promedio, máximo, conteo) a un modelo que lea la **secuencia** de movimientos
de una tarjeta. La sospecha del área de riesgos es concreta: hay patrones de fraude que solo se ven en el
orden — montos pequeños de prueba seguidos de una escalada rápida.

Usted va a construir, desde cero, la pieza que hace posible ese salto: una celda recurrente. Primero a mano
con NumPy, después verificada contra PyTorch, después una LSTM completa. Y al final va a **medir** hasta
qué distancia en el tiempo alcanza a aprender cada arquitectura.

### Cómo se trabaja

| Bloque | Qué va a hacer | Puntos |
|:--|:--|--:|
| 0 | **Investigar.** Lectura guiada y preguntas cortas, antes de escribir código. | 8 |
| 1–3 | **Desarrollar.** La celda recurrente, la salida y el alcance de la memoria, con NumPy. | 33 |
| 4 | **Verificar.** Ejecutar y comparar contra `torch.nn.RNN`. | 6 |
| 5 | **Desarrollar.** Una celda LSTM completa, a mano. | 14 |
| 6 | **Desarrollar y entrenar.** Una LSTM real sobre secuencias de transacciones. | 12 |
| 7 | **Investigar con evidencia.** ¿Hasta cuántos pasos atrás aprende cada arquitectura? | 14 |
| 8–9 | Análisis y conclusión, en sus propias palabras. | 13 |

**Todo se responde y se entrega dentro de este mismo notebook, con las celdas ejecutadas.**
Las celdas de `assert` son su verificación: si imprimen `OK`, el bloque está bien. Si fallan,
corrija antes de avanzar — los bloques posteriores dependen de los anteriores.

**Semanas:** 4 y 5 (laboratorio de dos semanas) · **Entrega:** viernes de la semana 5

In [1]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)
print("NumPy:", np.__version__)

NumPy: 2.3.5


---
## Bloque 0 — Investigación previa (8 puntos)

Antes de escribir código, lea:

- **PyTorch — `torch.nn.RNN`**: https://pytorch.org/docs/stable/generated/torch.nn.RNN.html
- **PyTorch — `torch.nn.LSTM`**: https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html
- **Understanding LSTM Networks** (Christopher Olah): https://colah.github.io/posts/2015-08-Understanding-LSTMs/

Responda con **sus propias palabras**, 2 a 4 líneas cada una. No copie de la documentación.

1. `nn.RNN` recibe tensores con forma `(seq_len, batch, input_size)` salvo que se pase `batch_first=True`.
   ¿Qué representa cada una de esas tres dimensiones, y por qué el valor por omisión es una fuente
   frecuente de errores silenciosos?
2. `nn.LSTM` devuelve `salida, (h_n, c_n)`. ¿Qué contiene cada uno de los tres, y cuál usaría para
   clasificar **la secuencia completa** como fraude o no fraude?
3. En la publicación de Olah, ¿qué papel cumple el **estado de celda** ($c_t$) y por qué el autor lo
   dibuja como una línea horizontal que atraviesa toda la celda sin ser modificada por completo?
4. ¿Qué significa que los parámetros de una RNN sean **compartidos en el tiempo**, y qué consecuencia
   práctica tiene eso para procesar clientes con historiales de largos distintos?

### Sus respuestas — Bloque 0

1. `seq_len` es la cantidad de pasos de la secuencia, `batch` es la cantidad de clientes que se procesan juntos e `input_size` es la cantidad de variables que trae cada paso. El problema del valor por omisión es que si yo preparo los datos con batch primero, PyTorch puede interpretar las dimensiones al revés sin dar un error claro.

2. `salida` guarda el estado oculto de la última capa en cada paso. `h_n` guarda el estado oculto final y `c_n` guarda el estado de celda final. Para clasificar toda la secuencia usaría `h_n[-1]` porque resume lo que la LSTM terminó guardando después de leer todos los movimientos.

3. El estado de celda funciona como una memoria que puede llevar información por varios pasos. La línea horizontal representa que esa información puede seguir avanzando casi sin cambios y que las compuertas deciden qué se borra, qué se agrega y qué se usa como salida.

4. Significa que la misma RNN usa los mismos pesos en todos los pasos de tiempo. Por eso no necesita aprender parámetros nuevos cuando una secuencia es más larga y puede trabajar con historiales de distintos largos, aunque al juntarlos en un batch sí hay que acomodar sus longitudes.


---
## Bloque 1 — La celda recurrente, a mano (15 puntos)

Esta es la secuencia con la que trabajaremos todo el laboratorio: cuatro movimientos de una misma tarjeta,
con el monto ya normalizado contra el gasto típico del cliente.

$$h_t = \tanh\!\left(w_x\,x_t + w_h\,h_{t-1} + b\right), \qquad h_0 = 0$$

Los pesos no están entrenados: son un punto de partida fijo para que todos obtengan los mismos números.

In [2]:
secuencia = np.array([0.2, 0.3, 0.9, 0.8])   # 4 movimientos, monto normalizado
w_x, w_h, b = 0.8, 0.5, 0.0                  # peso de la entrada, de la memoria, sesgo
h0 = 0.0

print("secuencia:", secuencia, " T =", len(secuencia))

secuencia: [0.2 0.3 0.9 0.8]  T = 4


Implemente la celda. Debe devolver **todos** los estados ocultos, no solo el último: los vamos a
necesitar en los bloques siguientes.

> Pista: un solo ciclo `for` sobre los pasos de tiempo. Guarde cada `h` en el arreglo de salida antes de
> pasar al siguiente paso. No use `np.tanh` sobre el vector completo de una vez — eso sería ignorar la
> recurrencia.

In [3]:
def celda_rnn(x, wx=w_x, wh=w_h, sesgo=b, h_inicial=h0):
    """Ejecuta una celda recurrente escalar sobre la secuencia x.

    Parámetros
    ----------
    x : np.ndarray de forma (T,)

    Retorna
    -------
    np.ndarray de forma (T,) con h_1, h_2, ..., h_T
    """
    T = len(x)
    estados = np.zeros(T)
    h = h_inicial

    for t in range(T):
        h = np.tanh(wx * x[t] + wh * h + sesgo)
        estados[t] = h

    return estados


h_seq = celda_rnn(secuencia)
print("estados ocultos:", h_seq)


estados ocultos: [0.1586 0.3089 0.7036 0.7581]


In [4]:
esperado_h = np.array([0.158649, 0.308896, 0.703627, 0.758135])

assert h_seq is not None, "celda_rnn devolvió None"
assert h_seq.shape == (4,), f"forma incorrecta: {h_seq.shape}, se esperaba (4,)"
assert np.allclose(h_seq, esperado_h, atol=1e-5), "los estados no coinciden con la referencia"
print("OK — Bloque 1")
print(f"h_4 = {h_seq[-1]:.6f}  <- este número resume las cuatro transacciones")

OK — Bloque 1
h_4 = 0.758135  <- este número resume las cuatro transacciones


**Pregunta rápida:** en $t_4$ la transacción **bajó** (de 0.9 a 0.8) y sin embargo el estado oculto
**subió** (de 0.704 a 0.758). Explique el mecanismo exacto que produce eso.

Aunque el monto bajó, la RNN no usa solo la transacción actual. En el paso anterior quedó un estado oculto alto por el monto de 0.9, así que en el siguiente paso se suma esa memoria al aporte del 0.8. Esa suma antes de aplicar `tanh` es mayor que la del paso anterior y por eso el estado oculto sube.


---
## Bloque 2 — El veredicto, y la prueba del orden (10 puntos)

La capa de salida es una neurona común que recibe $h_T$ en vez de datos crudos:

$$\hat y = \sigma\!\left(w_y\,h_T + b_y\right), \qquad w_y = 2.0,\; b_y = -1.0$$

Calcule la predicción para la secuencia original **y** para la misma secuencia invertida. Los cuatro montos
son idénticos: solo cambia el orden.

In [5]:
def sigmoide(z):
    return 1.0 / (1.0 + np.exp(-z))


def predecir(x, wy=2.0, by=-1.0):
    """Corre la celda recurrente sobre x y devuelve la probabilidad de fraude."""
    h_final = celda_rnn(x)[-1]
    return sigmoide(wy * h_final + by)


yhat_orden = predecir(secuencia)
yhat_invertida = predecir(secuencia[::-1])

print(f"orden original : {yhat_orden}")
print(f"orden invertido: {yhat_invertida}")


orden original : 0.6262749356316862
orden invertido: 0.4552897131009345


In [6]:
assert yhat_orden is not None and yhat_invertida is not None, "predecir devolvió None"
assert abs(yhat_orden - 0.626275) < 1e-5, f"predicción original incorrecta: {yhat_orden}"
assert abs(yhat_invertida - 0.455290) < 1e-5, f"predicción invertida incorrecta: {yhat_invertida}"

promedio = secuencia.mean()
maximo = secuencia.max()
assert np.isclose(promedio, secuencia[::-1].mean()) and np.isclose(maximo, secuencia[::-1].max())
print("OK — Bloque 2")
print(f"El modelo de variables agregadas ve exactamente lo mismo en ambos casos:")
print(f"  promedio = {promedio:.2f} | máximo = {maximo:.2f}")
print(f"La RNN ve dos casos distintos: {yhat_orden:.3f} vs {yhat_invertida:.3f}")

OK — Bloque 2
El modelo de variables agregadas ve exactamente lo mismo en ambos casos:
  promedio = 0.55 | máximo = 0.90
La RNN ve dos casos distintos: 0.626 vs 0.455


**Pregunta rápida:** describa una situación **real** de tarjeta de crédito en la que estos mismos cuatro
montos, en un orden, sean sospechosos y, en el otro, perfectamente normales.

Un caso sospechoso sería ver primero dos compras pequeñas de prueba y después dos cargos mucho más altos, porque parece que alguien probó la tarjeta antes de gastar más. En el orden contrario podría ser una persona que hizo dos compras grandes normales y después solo hizo gastos pequeños. Los montos son los mismos, pero la historia que cuenta el orden cambia.


---
## Bloque 3 — ¿Cuánto recuerda, en realidad? (8 puntos)

Vamos a medirlo, no a suponerlo. La influencia del paso $t$ sobre el estado final se puede aproximar
numéricamente: se perturba **una sola** entrada y se observa cuánto cambia $h_T$.

$$\text{influencia}(t) \;\approx\; \frac{\left|h_T(x + \varepsilon e_t) - h_T(x)\right|}{\varepsilon}$$

Implemente esa medición sobre una secuencia larga de 20 pasos.

In [7]:
secuencia_larga = np.full(20, 0.5)
epsilon = 1e-4


def influencia_por_paso(x, eps=epsilon):
    """Devuelve un arreglo de forma (T,) con la influencia de cada paso sobre h_T."""
    base = celda_rnn(x)[-1]
    infl = np.zeros(len(x))

    for t in range(len(x)):
        x_perturbado = x.copy()
        x_perturbado[t] += eps
        h_perturbado = celda_rnn(x_perturbado)[-1]
        infl[t] = abs(h_perturbado - base) / eps

    return infl


infl = influencia_por_paso(secuencia_larga)
print("influencia del primer paso :", infl[0])
print("influencia del último paso :", infl[-1])


influencia del primer paso : 2.6201263381153694e-10
influencia del último paso : 0.5058057644258263


In [8]:
assert infl is not None and infl.shape == (20,), "influencia_por_paso: forma incorrecta"
assert infl[-1] > infl[0], "el último paso debería influir más que el primero"
razon = infl[-1] / max(infl[0], 1e-12)
assert razon > 1000, f"la caída esperada es de varios órdenes de magnitud, obtuvo {razon:.1f}x"
print("OK — Bloque 3")
print(f"El último paso influye {razon:,.0f} veces más que el primero.")
print("Ese cociente ES el desvanecimiento del gradiente, medido.")

OK — Bloque 3
El último paso influye 1,930,463,265 veces más que el primero.
Ese cociente ES el desvanecimiento del gradiente, medido.


**Pregunta rápida:** con estos números, ¿a partir de qué paso diría que la red "ya no recuerda"?
Defina un criterio numérico explícito (por ejemplo, un umbral relativo) y aplíquelo.

Tomaría como criterio que la red ya no recuerda un paso cuando su influencia es menor al 1 por ciento de la influencia del último paso. Con los valores obtenidos, el paso 16 ya queda apenas por debajo de ese límite y todos los anteriores tienen todavía menos influencia. Entonces diría que desde cuatro pasos hacia atrás la memoria ya se vuelve muy débil con este criterio.


---
## Bloque 4 — Verificación contra PyTorch (6 puntos)

**Este bloque no lo desarrolla: lo ejecuta.** El código ya está escrito. Su trabajo es correrlo, entender
cómo se cargan los pesos en `nn.RNN` y responder la pregunta del final.

Observe con cuidado las formas de los tensores: es exactamente lo que investigó en el Bloque 0.

In [9]:
import torch
import torch.nn as nn

rnn = nn.RNN(input_size=1, hidden_size=1, num_layers=1, nonlinearity="tanh",
             bias=True, batch_first=True)

with torch.no_grad():                       # cargamos NUESTROS pesos, no los aleatorios
    rnn.weight_ih_l0.copy_(torch.tensor([[w_x]]))
    rnn.weight_hh_l0.copy_(torch.tensor([[w_h]]))
    rnn.bias_ih_l0.copy_(torch.tensor([b]))
    rnn.bias_hh_l0.copy_(torch.tensor([0.0]))

x_t = torch.tensor(secuencia, dtype=torch.float32).reshape(1, 4, 1)   # (batch, seq, features)
salida, h_n = rnn(x_t)

print("forma de la entrada:", tuple(x_t.shape))
print("forma de la salida :", tuple(salida.shape), " h_n:", tuple(h_n.shape))
print("\nPyTorch  :", salida.detach().numpy().squeeze())
print("Su NumPy :", h_seq)
print("\n¿Coinciden?", np.allclose(salida.detach().numpy().squeeze(), h_seq, atol=1e-5))

forma de la entrada: (1, 4, 1)
forma de la salida : (1, 4, 1)  h_n: (1, 1, 1)

PyTorch  : [0.1586 0.3089 0.7036 0.7581]
Su NumPy : [0.1586 0.3089 0.7036 0.7581]

¿Coinciden? True


**Pregunta:** `nn.RNN` tiene **dos** sesgos (`bias_ih_l0` y `bias_hh_l0`) mientras que nuestra ecuación
tiene uno solo. ¿Por qué esa diferencia no cambia lo que la red puede representar? ¿Y por qué, aun así,
PyTorch los mantiene separados?

Los dos sesgos de PyTorch se suman antes de aplicar la activación, así que juntos pueden hacer exactamente el papel de un solo sesgo. Por eso tenerlos separados no aumenta lo que la red puede representar. PyTorch los mantiene así porque separa los parámetros del camino de entrada y del camino recurrente, igual que hace con sus pesos.


---
## Bloque 5 — Una celda LSTM, a mano (14 puntos)

Ahora la celda que resolvió el problema del Bloque 3. Recuerde las cinco ecuaciones:

$$f_t = \sigma(\cdot) \quad i_t = \sigma(\cdot) \quad g_t = \tanh(\cdot) \quad o_t = \sigma(\cdot)$$

$$c_t = f_t \odot c_{t-1} + i_t \odot g_t \qquad h_t = o_t \odot \tanh(c_t)$$

Para poder seguir las cuentas a mano, las cuatro compuertas ya vienen calculadas — son los valores que
tendría una LSTM entrenada al llegar al paso $t_3$, la transacción de 0.9 que rompe el patrón.

In [10]:
# Compuertas ya calculadas para el paso t3 (una LSTM entrenada las produciría así)
f3, i3, g3, o3 = 0.2, 0.9, 0.8, 0.7
c2 = 0.35     # el "expediente" que venía del paso anterior

print(f"olvido f={f3} | entrada i={i3} | candidato g={g3} | salida o={o3} | c anterior={c2}")

olvido f=0.2 | entrada i=0.9 | candidato g=0.8 | salida o=0.7 | c anterior=0.35


In [11]:
def paso_lstm(c_anterior, f, i, g, o):
    """Un paso de celda LSTM.

    Retorna
    -------
    (c_nuevo, h_nuevo)
    """
    c_nuevo = f * c_anterior + i * g
    h_nuevo = o * np.tanh(c_nuevo)
    return c_nuevo, h_nuevo


c3, h3_lstm = paso_lstm(c2, f3, i3, g3, o3)
print(f"c3 = {c3}")
print(f"h3 = {h3_lstm}")


c3 = 0.79
h3 = 0.46088632516867567


In [12]:
assert c3 is not None and h3_lstm is not None, "paso_lstm devolvió None"
assert abs(c3 - 0.79) < 1e-6, f"c3 incorrecto: {c3}"
assert abs(h3_lstm - 0.460886) < 1e-5, f"h3 incorrecto: {h3_lstm}"
print("OK — Bloque 5")
print(f"La RNN simple, en ese mismo paso, dio h3 = {h_seq[2]:.3f}")
print(f"La LSTM reporta menos ({h3_lstm:.3f}) pero guarda más (c3 = {c3:.2f}).")

OK — Bloque 5
La RNN simple, en ese mismo paso, dio h3 = 0.704
La LSTM reporta menos (0.461) pero guarda más (c3 = 0.79).


Ahora la parte que importa. En la RNN simple, el eslabón de la cadena hacia atrás valía
$w_h(1-h^2)$ — un número que la red **no controla**. En la LSTM vale $f_t$, un número que **aprende**.

Compare los dos, sobre 40 pasos.

In [13]:
def factor_cadena_rnn(h, wh=w_h):
    """Eslabón de la cadena de la RNN simple: dh_t/dh_{t-1}."""
    return wh * (1 - h ** 2)


pasos = np.arange(1, 41)
eslabon_rnn = factor_cadena_rnn(h_seq[-1])
cadena_rnn = eslabon_rnn ** pasos
cadena_lstm = 0.95 ** pasos

print("a 40 pasos — RNN :", cadena_rnn[-1] if cadena_rnn is not None else None)
print("a 40 pasos — LSTM:", cadena_lstm[-1] if cadena_lstm is not None else None)


a 40 pasos — RNN : 1.270103231203763e-27
a 40 pasos — LSTM: 0.12851215656510312


In [14]:
assert cadena_rnn is not None and cadena_lstm is not None, "falta calcular alguna cadena"
assert cadena_lstm[-1] / cadena_rnn[-1] > 1e6, "la LSTM debería conservar órdenes de magnitud más"
print("OK — Bloque 5b")
print(f"La LSTM conserva {cadena_lstm[-1]/cadena_rnn[-1]:.2e} veces más gradiente a 40 pasos.")

OK — Bloque 5b
La LSTM conserva 1.01e+26 veces más gradiente a 40 pasos.


---
## Bloque 6 — Una LSTM real sobre transacciones (12 puntos)

Los datos se generan aquí mismo, de forma determinista, para que el laboratorio funcione sin conexión.
Cada cliente tiene 30 movimientos. Los casos de fraude tienen un patrón concreto: **una escalada sostenida
al final de la secuencia**. Los normales tienen los mismos montos, pero desordenados.

La celda de datos ya está escrita: ejecútela y observe la forma del tensor.

In [15]:
import torch
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
rng = np.random.default_rng(42)

N, T = 2000, 30


def generar_datos(n=N, largo=T, semilla=7):
    r = np.random.default_rng(semilla)
    X = r.uniform(0.05, 0.45, size=(n, largo))
    y = np.zeros(n)
    idx_fraude = r.choice(n, size=n // 2, replace=False)
    escalada = np.linspace(0.5, 0.95, 6)
    for i in idx_fraude:
        X[i, -6:] = escalada + r.normal(0, 0.03, size=6)   # escalada al final
        y[i] = 1.0
    # los normales reciben los MISMOS montos altos, pero dispersos al azar
    for i in set(range(n)) - set(idx_fraude):
        posiciones = r.choice(largo, size=6, replace=False)
        X[i, posiciones] = escalada + r.normal(0, 0.03, size=6)
    return X.astype(np.float32), y.astype(np.float32)


X, y = generar_datos()
X_ent, y_ent = X[:1600], y[:1600]
X_val, y_val = X[1600:], y[1600:]

ds = TensorDataset(torch.tensor(X_ent).unsqueeze(-1), torch.tensor(y_ent).unsqueeze(-1))
cargador = DataLoader(ds, batch_size=64, shuffle=True)

print("forma del tensor de entrada:", tuple(next(iter(cargador))[0].shape), " <- (batch, seq, features)")
print("proporción de fraude:", y_ent.mean())
print("promedio por clase (fraude / normal):", X_ent[y_ent == 1].mean().round(4),
      "/", X_ent[y_ent == 0].mean().round(4), " <- casi idénticos: el orden es la única señal")

forma del tensor de entrada: (64, 30, 1)  <- (batch, seq, features)
proporción de fraude: 0.499375
promedio por clase (fraude / normal): 0.3462 / 0.3451  <- casi idénticos: el orden es la única señal


Complete el modelo. Debe recibir `(batch, 30, 1)` y devolver **un logit por secuencia**.

> Pista: `nn.LSTM(input_size=1, hidden_size=32, batch_first=True)` devuelve `salida, (h_n, c_n)`.
> Para clasificar la secuencia completa, use `h_n[-1]` — no `salida`.

In [16]:
class DetectorFraude(nn.Module):
    def __init__(self, oculto=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=oculto, batch_first=True)
        self.salida = nn.Linear(oculto, 1)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return self.salida(h_n[-1])


modelo = DetectorFraude()
prueba = modelo(torch.tensor(X_val[:8]).unsqueeze(-1))
print("forma de la salida:", tuple(prueba.shape))


forma de la salida: (8, 1)


In [17]:
assert prueba is not None, "el modelo devolvió None"
assert tuple(prueba.shape) == (8, 1), f"la salida debe ser (8, 1) y salió {tuple(prueba.shape)}"
print("OK — Bloque 6a")

OK — Bloque 6a


El bucle de entrenamiento ya está escrito. Ejecútelo (toma menos de un minuto en CPU).

In [18]:
def entrenar(modelo, cargador, epocas=6, lr=1e-2):
    criterio = nn.BCEWithLogitsLoss()
    opt = torch.optim.Adam(modelo.parameters(), lr=lr)
    for epoca in range(epocas):
        modelo.train()
        acum, n = 0.0, 0
        for xb, yb in cargador:
            opt.zero_grad()
            perdida = criterio(modelo(xb), yb)
            perdida.backward()
            opt.step()
            acum += perdida.item() * xb.size(0); n += xb.size(0)
        print(f"época {epoca+1}  pérdida {acum/n:.4f}")
    return modelo


def exactitud(modelo, X_, y_):
    modelo.eval()
    with torch.no_grad():
        p = torch.sigmoid(modelo(torch.tensor(X_).unsqueeze(-1))).numpy().ravel()
    return float(((p > 0.5) == (y_ > 0.5)).mean())


modelo = entrenar(modelo, cargador)
print("\nexactitud en validación:", round(exactitud(modelo, X_val, y_val), 4))

época 1  pérdida 0.4966
época 2  pérdida 0.1766
época 3  pérdida 0.0512


época 4  pérdida 0.0215
época 5  pérdida 0.0209
época 6  pérdida 0.0166

exactitud en validación: 0.985


Ahora la comparación que da sentido a todo el laboratorio: un modelo que **ignora el orden**.
Se le entregan solo tres variables agregadas — promedio, máximo y desviación — sobre la misma secuencia.

In [19]:
def variables_agregadas(X_):
    """Devuelve un arreglo (n, 3) con promedio, máximo y desviación de cada secuencia."""
    promedio = X_.mean(axis=1)
    maximo = X_.max(axis=1)
    desviacion = X_.std(axis=1)
    return np.column_stack([promedio, maximo, desviacion]).astype(np.float32)


Xa_ent = variables_agregadas(X_ent)
Xa_val = variables_agregadas(X_val)

modelo_agregado = nn.Sequential(nn.Linear(3, 32), nn.ReLU(), nn.Linear(32, 1))
opt = torch.optim.Adam(modelo_agregado.parameters(), lr=1e-2)
crit = nn.BCEWithLogitsLoss()
xt, yt = torch.tensor(Xa_ent, dtype=torch.float32), torch.tensor(y_ent).unsqueeze(-1)

for epoca in range(400):
    opt.zero_grad(); crit(modelo_agregado(xt), yt).backward(); opt.step()

with torch.no_grad():
    pv = torch.sigmoid(modelo_agregado(torch.tensor(Xa_val, dtype=torch.float32))).numpy().ravel()
exactitud_agregado = float(((pv > 0.5) == (y_val > 0.5)).mean())
print("exactitud del modelo de variables agregadas:", round(exactitud_agregado, 4))


exactitud del modelo de variables agregadas: 0.5075


In [20]:
assert Xa_ent is not None and Xa_ent.shape == (1600, 3), "variables_agregadas: forma incorrecta"
print("OK — Bloque 6b")
print(f"LSTM               : {exactitud(modelo, X_val, y_val):.4f}")
print(f"Variables agregadas: {exactitud_agregado:.4f}")

OK — Bloque 6b
LSTM               : 0.9850
Variables agregadas: 0.5075


**Pregunta rápida:** el modelo de variables agregadas ronda el azar. Explique, usando cómo se
generaron los datos, por qué era **imposible** que le fuera mejor.

Era muy difícil que ese modelo mejorara porque las dos clases reciben prácticamente los mismos montos altos y bajos. La diferencia real está en dónde aparecen: en fraude forman una escalada al final y en los casos normales están dispersos. Al quedarse solo con promedio, máximo y desviación se pierde justamente el orden que separa las clases.


---
## Bloque 7 — Reto de investigación (14 puntos)

Aquí no hay `TODO` que rellenar. Hay una pregunta, y usted tiene que **producir evidencia**. Se califica el
diseño del experimento y la interpretación, no que el número salga bonito.

### La pregunta

**¿Hasta qué distancia en el tiempo alcanza a aprender cada arquitectura?**

Diseñe una tarea donde la etiqueta dependa **exclusivamente** de un evento que ocurre a $k$ pasos del final:
por ejemplo, una transacción marcada en la posición $T-k$ que determina la clase, mientras el resto de la
secuencia es ruido idéntico en ambas clases.

Entrene, para al menos **cuatro** valores de $k$ (por ejemplo 2, 5, 15 y 30), estos dos modelos:

- una `nn.RNN` simple con `hidden_size=32`
- su `DetectorFraude` con LSTM, mismo tamaño

Reporte una tabla que usted construya:

| k (distancia) | exactitud RNN | exactitud LSTM |
|--:|--:|--:|
| 2 | | |
| 5 | | |
| 15 | | |
| 30 | | |

Y responda: **¿a partir de qué distancia la RNN simple deja de aprender, y coincide eso con lo que midió
en el Bloque 3?**

In [21]:
# Reto — experimento con una señal a k pasos del final

class DetectorRNN(nn.Module):
    def __init__(self, oculto=32):
        super().__init__()
        self.rnn = nn.RNN(input_size=1, hidden_size=oculto, batch_first=True)
        self.salida = nn.Linear(oculto, 1)

    def forward(self, x):
        _, h_n = self.rnn(x)
        return self.salida(h_n[-1])


def generar_datos_reto(k, n=2000, largo=40, semilla=100):
    r = np.random.default_rng(semilla + k)

    # Todo es ruido igual para ambas clases.
    X = r.normal(0.0, 0.1, size=(n, largo))
    y = np.zeros(n, dtype=np.float32)
    y[:n // 2] = 1.0
    r.shuffle(y)

    # La única señal de la clase está exactamente a k pasos del final.
    posicion = largo - k
    X[:, posicion] += 2.0 * y

    return X.astype(np.float32), y


def entrenar_reto(modelo, X_entreno, y_entreno, X_validacion, y_validacion,
                  epocas=12, lr=1e-2, semilla=1):
    generador = torch.Generator().manual_seed(semilla)
    ds_reto = TensorDataset(
        torch.tensor(X_entreno).unsqueeze(-1),
        torch.tensor(y_entreno).unsqueeze(-1)
    )
    cargador_reto = DataLoader(
        ds_reto,
        batch_size=64,
        shuffle=True,
        generator=generador
    )

    criterio = nn.BCEWithLogitsLoss()
    opt = torch.optim.Adam(modelo.parameters(), lr=lr)

    for _ in range(epocas):
        modelo.train()
        for xb, yb in cargador_reto:
            opt.zero_grad()
            perdida = criterio(modelo(xb), yb)
            perdida.backward()
            opt.step()

    modelo.eval()
    with torch.no_grad():
        logits = modelo(torch.tensor(X_validacion).unsqueeze(-1))
        pred = torch.sigmoid(logits).numpy().ravel()

    return float(((pred > 0.5) == (y_validacion > 0.5)).mean())


valores_k = [2, 5, 15, 30]
resultados_reto = []

for k in valores_k:
    Xr, yr = generar_datos_reto(k)
    Xr_ent, yr_ent = Xr[:1600], yr[:1600]
    Xr_val, yr_val = Xr[1600:], yr[1600:]

    torch.manual_seed(10100 + k)
    modelo_rnn_reto = DetectorRNN(oculto=32)
    acc_rnn = entrenar_reto(
        modelo_rnn_reto, Xr_ent, yr_ent, Xr_val, yr_val,
        semilla=20100 + k
    )

    torch.manual_seed(30100 + k)
    modelo_lstm_reto = DetectorFraude(oculto=32)
    acc_lstm = entrenar_reto(
        modelo_lstm_reto, Xr_ent, yr_ent, Xr_val, yr_val,
        semilla=40100 + k
    )

    resultados_reto.append((k, acc_rnn, acc_lstm))

print("k | exactitud RNN | exactitud LSTM")
print("--|---------------|---------------")
for k, acc_rnn, acc_lstm in resultados_reto:
    print(f"{k:2d}| {acc_rnn:13.4f} | {acc_lstm:14.4f}")


k | exactitud RNN | exactitud LSTM
--|---------------|---------------
 2|        1.0000 |         1.0000
 5|        1.0000 |         1.0000
15|        0.4950 |         1.0000
30|        0.4875 |         0.5300


### Tabla de resultados y conclusión del Bloque 7

| k | exactitud RNN | exactitud LSTM |
|--:|--:|--:|
| 2 | 1.0000 | 1.0000 |
| 5 | 1.0000 | 1.0000 |
| 15 | 0.4950 | 1.0000 |
| 30 | 0.4875 | 0.5300 |

La RNN simple deja de aprender claramente desde k = 15, donde cae prácticamente al azar, mientras que la LSTM todavía llega a una exactitud perfecta. En k = 30 las dos tienen problemas y la LSTM queda apenas arriba del azar. Esto sí coincide con la idea del Bloque 3 porque la RNN pierde muy rápido la influencia de información lejana, aunque no coincide en una distancia exacta porque aquí la RNN tiene 32 unidades y además sus pesos sí fueron entrenados.


---
## Bloque 8 — Análisis (8 puntos)

En sus propias palabras, 3 a 6 líneas cada una.

1. El área de riesgos quiere usar una LSTM **bidireccional** para bloquear transacciones en el momento de
   la autorización. Explique por qué eso es imposible, y qué le propondría en su lugar.
2. Un compañero normaliza los montos dividiendo entre el máximo **de todo el conjunto de datos**, incluidos
   los de validación. Explique qué tipo de error es ese y cómo afectaría los resultados que reportó hoy.
3. La LSTM de este laboratorio tiene una sola variable por paso (el monto). En producción cada transacción
   trae monto, hora, canal, país y comercio. ¿Qué cambia exactamente en el código, y qué **no** cambia?

### Sus respuestas — Bloque 8

1. Una LSTM bidireccional necesita leer también lo que viene después del punto que está evaluando. En una autorización real esas transacciones futuras todavía no existen, así que no se pueden usar para decidir en ese momento. Usaría una LSTM normal que lea solo el historial anterior y la transacción actual.

2. Eso es fuga de datos porque la validación está ayudando a definir cómo se transforman los datos antes de entrenar. El resultado puede verse mejor de lo que realmente sería con datos nuevos. El máximo debe calcularse solo con entrenamiento y luego esa misma escala se aplica a validación.

3. La entrada ya no tendría una sola variable por paso, así que `input_size` tendría que cambiar al número de variables que entren al modelo después de preparar los datos. También habría que convertir las variables de texto a números y escalar lo necesario. La idea de leer una secuencia y clasificar con el estado final no cambia.


---
## Bloque 9 — Conclusión (5 puntos)

Dos preguntas. Ninguna tiene respuesta correcta; sí tienen respuesta **honesta**.

1. ¿En qué punto exacto de este laboratorio algo dejó de tener sentido para usted, y qué hizo para salir de
   ahí? Si nada se le atoró, diga qué parte le resultó más fácil de lo que esperaba y por qué.
2. Usted midió el desvanecimiento del gradiente en el Bloque 3 antes de que la clase se lo explicara del
   todo. ¿Qué le dio medirlo que no le habría dado leerlo?

### Sus respuestas — Bloque 9

1. La parte que más me costó fue el Bloque 7 porque esperaba que la diferencia entre RNN y LSTM saliera clarísima de una vez y no pasó así en todas las distancias. Para entenderlo mejor revisé que la señal estuviera solo en el paso correcto, fijé las semillas y comparé los resultados sin asumir que una arquitectura tenía que ganar siempre.

2. Medirlo me ayudó a ver qué tan rápido se pierde de verdad la influencia de los primeros pasos. Leer que existe desvanecimiento del gradiente suena abstracto, pero ver que el último paso puede influir miles de millones de veces más que el primero hace que el problema se entienda mucho más fácil.
